In [0]:
import pandas as pd
import json
from pyspark.sql import functions as F

# Load claims as Spark DF, convert to pandas for function work
df_spark = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv",
    header=True,
    inferSchema=True
)
df = df_spark.toPandas()
print(f"Loaded {len(df)} rows")
df.head()

Loaded 10 rows


,claim_id,member_id,provider_id,service_date,admit_date,discharge_date,diagnosis_code,procedure_code,billed_amount,allowed_amount,paid_amount,claim_status,denial_reason,load_timestamp
0,1114,5,3,2026-06-01,2026-06-01,2026-06-03,J18.9,99213,4200,3800,3800,Approved,None,2026-06-22
1,1115,12,7,2026-06-03,2026-06-03,2026-06-03,M54.5,99214,950,800,800,Approved,None,2026-06-22
2,1116,19,2,2026-06-05,2026-06-05,2026-06-05,Z00.00,99385,350,0,0,Denied,Not Medically Necessary,2026-06-22
3,1117,8,10,2026-06-07,2026-06-07,2026-06-09,I10,99213,7800,7200,7200,Approved,None,2026-06-22
4,1118,3,5,2026-06-08,2026-06-08,2026-06-08,E11.9,99214,620,0,0,Denied,Prior Auth Required,2026-06-22


In [0]:
def check_completeness(df, column_name):
    """
    Returns % of non-null values in a column.
    SQL equivalent: SUM(CASE WHEN col IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*) * 100
    """
    total = len(df)
    non_null = df[column_name].notna().sum()
    score = round((non_null / total) * 100, 2)
    return {
        "check": "Completeness",
        "column": column_name,
        "total_rows": total,
        "non_null_rows": int(non_null),
        "score_pct": score
    }

# Test it
print(check_completeness(df, "claim_id"))
print(check_completeness(df, "procedure_code"))

{'check': 'Completeness', 'column': 'claim_id', 'total_rows': 10, 'non_null_rows': 10, 'score_pct': np.float64(100.0)}
{'check': 'Completeness', 'column': 'procedure_code', 'total_rows': 10, 'non_null_rows': 10, 'score_pct': np.float64(100.0)}


In [0]:
def check_uniqueness(df, column_name):
    """
    Returns % of distinct values vs total rows.
    SQL equivalent: COUNT(DISTINCT col) / COUNT(*) * 100
    """
    total = len(df)
    distinct = df[column_name].nunique()
    score = round((distinct / total) * 100, 2)
    return {
        "check": "Uniqueness",
        "column": column_name,
        "total_rows": total,
        "distinct_values": int(distinct),
        "score_pct": score
    }

print(check_uniqueness(df, "claim_id"))
print(check_uniqueness(df, "claim_status"))

{'check': 'Uniqueness', 'column': 'claim_id', 'total_rows': 10, 'distinct_values': 10, 'score_pct': 100.0}
{'check': 'Uniqueness', 'column': 'claim_status', 'total_rows': 10, 'distinct_values': 2, 'score_pct': 20.0}


In [0]:
def check_validity(df, column_name, valid_values):
    """
    Returns % of rows where column value is in an allowed set.
    SQL equivalent: SUM(CASE WHEN col IN (...) THEN 1 ELSE 0 END) / COUNT(*) * 100
    """
    total = len(df)
    valid_count = df[column_name].isin(valid_values).sum()
    score = round((valid_count / total) * 100, 2)
    return {
        "check": "Validity",
        "column": column_name,
        "valid_values": valid_values,
        "valid_rows": int(valid_count),
        "score_pct": score
    }

print(check_validity(df, "claim_status", ["Approved", "Denied", "Pending"]))

{'check': 'Validity', 'column': 'claim_status', 'valid_values': ['Approved', 'Denied', 'Pending'], 'valid_rows': 10, 'score_pct': np.float64(100.0)}


In [0]:
def check_integrity(df, fk_column, valid_ids):
    """
    Returns % of FK values that exist in a reference set.
    SQL equivalent: JOIN to reference table, flag unmatched rows
    """
    total = len(df)
    matched = df[fk_column].isin(valid_ids).sum()
    score = round((matched / total) * 100, 2)
    return {
        "check": "Integrity",
        "column": fk_column,
        "total_rows": total,
        "matched_rows": int(matched),
        "score_pct": score
    }

# Use your known member/provider IDs as the reference set
known_member_ids = list(df["member_id"].dropna().unique())
print(check_integrity(df, "member_id", known_member_ids))

{'check': 'Integrity', 'column': 'member_id', 'total_rows': 10, 'matched_rows': 10, 'score_pct': np.float64(100.0)}


In [0]:
def validate_json_schema(json_path, required_fields):
    """
    Reads a JSON file from DBFS volume, validates each record
    has all required fields. Flags missing fields per record.
    """
    with open(json_path, "r") as f:
        records = [json.loads(line) for line in f if line.strip()]
    
    results = []
    for i, record in enumerate(records):
        missing = [field for field in required_fields if field not in record]
        results.append({
            "record_index": i,
            "has_all_fields": len(missing) == 0,
            "missing_fields": missing if missing else "None"
        })
    
    valid_count = sum(1 for r in results if r["has_all_fields"])
    print(f"Validated {len(records)} records — {valid_count} passed, {len(records)-valid_count} failed")
    return pd.DataFrame(results)

# Run against your existing JSON file
required = ["claim_id", "member_id", "provider_id", "billed_amount", "claim_status"]
json_results = validate_json_schema(
    "/Volumes/workspace/default/hc_practice_data/claims_data.json",
    required
)
json_results

Validated 10 records — 10 passed, 0 failed


,record_index,has_all_fields,missing_fields
0,0,True,None
1,1,True,None
2,2,True,None
3,3,True,None
4,4,True,None
5,5,True,None
6,6,True,None
7,7,True,None
8,8,True,None
9,9,True,None


In [0]:
# Run all 4 metric functions and compile into one DataFrame
results = [
    check_completeness(df, "claim_id"),
    check_completeness(df, "procedure_code"),
    check_uniqueness(df, "claim_id"),
    check_validity(df, "claim_status", ["Approved", "Denied", "Pending"]),
    check_integrity(df, "member_id", known_member_ids)
]

quality_report = pd.DataFrame(results)[["check", "column", "score_pct"]]
quality_report.columns = ["Check Type", "Column", "Score (%)"]
quality_report["Pass"] = quality_report["Score (%)"] >= 95.0

print("=== HealthcarePractice Data Quality Report ===")
quality_report

=== HealthcarePractice Data Quality Report ===


,Check Type,Column,Score (%),Pass
0,Completeness,claim_id,100.0,True
1,Completeness,procedure_code,100.0,True
2,Uniqueness,claim_id,100.0,True
3,Validity,claim_status,100.0,True
4,Integrity,member_id,100.0,True
